# About HSDS

HSDS (HDF Scalable Data Service) is a REST-based service desgined for the reading and writing of HDF5 data.  In HSDS traditional HDF5
files are stored as sharded collections of JSON objects (for metadata) and binary blobs (for chunk data).  This storage layout enables efficient reads and writes to AWS S3 stores (though POSIX storage is also supported).  HSDS runs as a set of front-end (the SN nodes)
and back-end (the DN nodes) processes.  The SN nodes provide horizontal scaling (more SN nodes can support more clients) while the DN
nodes are used to parallize individual client requests.  The number of SN and DN nodes can be configured at start up time subject
to the available memory and CPU resources of the machine.

HSDS clients can use the REST API directly are use the h5pyd package (for Python) or the REST VOL (for C/C++).  This study has 
focused on Python clients using h5pyd.

# Comparison of old vs new HSDS release when Accessing Single-Shot HDF5 files in AWS S3

HSDS was initially designed to support files that contained relatively few (but potentially quite large) datasets.  By contrast,
the data collected from CMOD, contains a large number of smaller datasets (i.e. signals).  This resulted in poor performance due to
each read or write of a signal requiring an seperate HTTP requests.

The forthcoming HSDS 1.0 (as of writing this) along with the corresponding h5pyd client library improves performance
by batching multiple operations in a single request, thus reducing latency.  Write performance is improved by enabling multiple updates (e.g. creating mulitple datasets) to be done in a single http requests.  Read performance is improved by the use of <I>consolidated metadata objects</I>.  These are JSON objects that contain all the metadata for a given file.  HSDS will create these objects in a background task when the file has not been modified recently.  On file open, the h5pyd client will retrieve the consolidated metadata (if available), cache it and use it to return results for any future api read operations that don't require dataset data.

This notebook compares benchmark results between the latest current HSDS release and a development version of future 1.0 release. The goal is to assess whether there are significant performance differences between the old and new code.

The test data consists of 100 C-mod HDF5 files that were imported into HSDS using the hsload utility. 

HSDS was configured with 4 SN and 4 DN nodes using an S3 bucket as the data store.

## TL;DR Conclusions

* Importing data using hsload was approximately 2x faster with the new HSDS version
* With one worker, performance improved by 30%
* With more workers, the improvement was greater with a 82% improvement using 16 workers
* Beyond 16 workers, HSDS (both the old and the new versions) with 4 SN and 4 DN nodes was not able to handle the client load and the test failed

The overall conclusion is the new HSDS release is a signficant improvement for working with files that contain many smaller objects.  The new design reduces latency by sending more information per request, so that the total number of requests needed is less.  The improvement is even greater as the number of workers increases (up to a point).  With more than 16 workers, the HSDS configuration would need to be scaled up (say to 8 SN and 8 DN nodes), but the system we ran the test on was limited to 8 CPU cores and 64 GB of memory.

---

## Benchmark Data Analysis

In [2]:
import pandas as pd
import hvplot.pandas  # noqa: F401
import holoviews as hv

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

hv.extension("bokeh")

Read benchmark data for the new HSDS service:

In [3]:
s3_data_new = pd.read_csv("./bench.hsdsv1.csv")
s3_data_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 5830 entries, 0 to 5829
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   worker#              5830 non-null   int64  
 1   obj-id               5830 non-null   str    
 2   open+read-data-time  5830 non-null   float64
 3   wrkr-num-objs        5830 non-null   int64  
 4   mean-obj-time        5830 non-null   float64
 5   num-dsets            5830 non-null   int64  
 6   mean-dset-time       5830 non-null   float64
 7   num-workers          5830 non-null   int64  
 8   obj-type             5830 non-null   str    
 9   tot-num-obj          5830 non-null   int64  
 10  total-runtime        5830 non-null   float64
dtypes: float64(4), int64(5), str(2)
memory usage: 582.5 KB


Read benchmark data for the old HSDS:

In [8]:
s3_data_old = pd.read_csv("./bench.hsdsv09.csv")
s3_data_old.info()

<class 'pandas.DataFrame'>
RangeIndex: 5830 entries, 0 to 5829
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   worker#              5830 non-null   int64  
 1   obj-id               5830 non-null   str    
 2   open+read-data-time  5830 non-null   float64
 3   wrkr-num-objs        5830 non-null   int64  
 4   mean-obj-time        5830 non-null   float64
 5   num-dsets            5830 non-null   int64  
 6   mean-dset-time       5830 non-null   float64
 7   num-workers          5830 non-null   int64  
 8   obj-type             5830 non-null   str    
 9   tot-num-obj          5830 non-null   int64  
 10  total-runtime        5830 non-null   float64
dtypes: float64(4), int64(5), str(2)
memory usage: 582.5 KB


In [9]:
s3_data_old.groupby(["num-workers"]).describe()

worker#                                                   \
              count      mean       std  min   25%  50%    75%   max   
num-workers                                                            
1               1.0  1.000000       NaN  1.0  1.00  1.0   1.00   1.0   
2               2.0  1.500000  0.707107  1.0  1.25  1.5   1.75   2.0   
4               4.0  2.500000  1.290994  1.0  1.75  2.5   3.25   4.0   
8            2001.0  4.488756  2.287793  1.0  2.00  4.0   6.00   8.0   
16           3822.0  8.131868  4.415645  1.0  4.00  8.0  12.00  16.0   

            open+read-data-time                                              \
                          count       mean        std        min        25%   
num-workers                                                                   
1                           1.0  79.615679        NaN  79.615679  79.615679   
2                           2.0  40.406347   8.552716  34.358664  37.382505   
4                           4.0  23.600481   4.413485  17.154838  22.872143   
8                        2001.0  19.992221   6.867547   1.734124  16.818004   
16                       3822.0  25.923089  14.964076   1.185076  18.013025   

                                              wrkr-num-objs             \
                   50%        75%         max         count       mean   
num-workers                                                              
1            79.615679  79.615679   79.615679           1.0   1.000000   
2            40.406347  43.430188   46.454030           2.0   1.000000   
4            25.055297  25.783635   27.136491           4.0   1.000000   
8            18.696825  21.985912   69.079080        2001.0  12.104448   
16           23.056537  30.235570  276.591591        3822.0   6.472004   

                                                   mean-obj-time             \
                  std  min   25%   50%   75%   max         count       mean   
num-workers                                                                   
1                 NaN  1.0   1.0   1.0   1.0   1.0           1.0  79.615679   
2            0.000000  1.0   1.0   1.0   1.0   1.0           2.0  40.406347   
4            0.000000  1.0   1.0   1.0   1.0   1.0           4.0  23.600481   
8            2.088441  1.0  12.0  13.0  13.0  14.0        2001.0   1.686896   
16           1.339757  1.0   7.0   7.0   7.0   8.0        3822.0   4.017112   

                                                                              \
                  std        min        25%        50%        75%        max   
num-workers                                                                    
1                 NaN  79.615679  79.615679  79.615679  79.615679  79.615679   
2            8.552716  34.358664  37.382505  40.406347  43.430188  46.454030   
4            4.413485  17.154838  22.872143  25.055297  25.783635  27.136491   
8            0.848187   0.867062   1.364915   1.498993   1.758538  16.563525   
16           2.160517   0.978687   2.772533   3.490580   4.513129  39.513084   

            num-dsets                                                       \
                count        mean        std    min     25%    50%     75%   
num-workers                                                                  
1                 1.0  785.000000        NaN  785.0  785.00  785.0  785.00   
2                 2.0  393.500000  30.405592  372.0  382.75  393.5  404.25   
4                 4.0  198.750000  32.190837  152.0  190.25  211.0  219.50   
8              2001.0   24.382309   8.055698    2.0   24.00   26.0   26.00   
16             3822.0   13.021716   4.446230    1.0   12.00   14.0   14.00   

                   mean-dset-time                                          \
               max          count      mean       std       min       25%   
num-workers                                                                 
1            785.0            1.0  0.101421       NaN  0.101421  0.101421   
2 

In [10]:
s3_data_new.groupby(["num-workers"]).describe()


worker#                                                   \
              count      mean       std  min   25%  50%    75%   max   
num-workers                                                            
1               1.0  1.000000       NaN  1.0  1.00  1.0   1.00   1.0   
2               2.0  1.500000  0.707107  1.0  1.25  1.5   1.75   2.0   
4               4.0  2.500000  1.290994  1.0  1.75  2.5   3.25   4.0   
8            2001.0  4.488756  2.287793  1.0  2.00  4.0   6.00   8.0   
16           3822.0  8.131868  4.415645  1.0  4.00  8.0  12.00  16.0   

            open+read-data-time                                              \
                          count       mean        std        min        25%   
num-workers                                                                   
1                           1.0  55.792627        NaN  55.792627  55.792627   
2                           2.0  30.570720  11.806563  22.222220  26.396470   
4                           4.0   1.151833   0.501333   0.691516   0.852922   
8                        2001.0   6.471680   3.585904   0.616644   4.094138   
16                       3822.0   4.470909   2.144953   0.371542   3.387457   

                                             wrkr-num-objs             \
                   50%        75%        max         count       mean   
num-workers                                                             
1            55.792627  55.792627  55.792627           1.0   1.000000   
2            30.570720  34.744971  38.919221           2.0   1.000000   
4             1.034894   1.333805   1.846029           4.0   1.000000   
8             5.160867   8.166348  41.816085        2001.0  12.104448   
16            4.386515   5.412809  30.171380        3822.0   6.472004   

                                                   mean-obj-time             \
                  std  min   25%   50%   75%   max         count       mean   
num-workers                                                                   
1                 NaN  1.0   1.0   1.0   1.0   1.0           1.0  55.792627   
2            0.000000  1.0   1.0   1.0   1.0   1.0           2.0  30.570720   
4            0.000000  1.0   1.0   1.0   1.0   1.0           4.0   1.151833   
8            2.088441  1.0  12.0  13.0  13.0  14.0        2001.0   0.550240   
16           1.339757  1.0   7.0   7.0   7.0   8.0        3822.0   0.697789   

                                                                               \
                   std        min        25%        50%        75%        max   
num-workers                                                                     
1                  NaN  55.792627  55.792627  55.792627  55.792627  55.792627   
2            11.806563  22.222220  26.396470  30.570720  34.744971  38.919221   
4             0.501333   0.691516   0.852922   1.034894   1.333805   1.846029   
8             0.392553   0.220715   0.325456   0.538035   0.645994   7.501430   
16            0.353465   0.231105   0.527890   0.656601   0.803128   6.165128   

            num-dsets                                                       \
                count        mean        std    min     25%    50%     75%   
num-workers                                                                  
1                 1.0  776.000000        NaN  776.0  776.00  776.0  776.00   
2                 2.0  402.500000  41.719300  373.0  387.75  402.5  417.25   
4                 4.0  194.000000  22.963740  162.0  185.25  200.5  209.25   
8              2001.0   24.376312   7.967673    2.0   24.00   26.0   26.00   
16             3822.0   13.024333   4.449041    1.0   12.00   14.0   14.00   

                   mean-dset-time                                          \
               max          count      mean       std       min       25%   
num-workers                                                                 
1            776.0            1.0  0.071898       NaN  0.071898  0.071898   
2 

These were the benchmark parameter combinations:

In [6]:
s3_data_old[["obj-type", "num-workers"]].drop_duplicates()

,obj-type,num-workers
0,shots,1
1,shots,2
3,shots,4
7,signals,8
2000,shots,8
2008,signals,16
5814,shots,16


In [11]:
s3_data_new[["obj-type",  "num-workers"]].drop_duplicates()

,obj-type,num-workers
0,shots,1
1,shots,2
3,shots,4
7,signals,8
2000,shots,8
2008,signals,16
5814,shots,16


Column `obj-type` describes data access type during one benchmark run:

* `obj-type = shots` means all signals from one shot file were read.
* `obj-type = signals` means all signals from all the files were read, one at a time.


Since the two ways of reading data by `shots` and `signals` are so different they cannot be compared to each other. Separate them into different DataFrames.

In [12]:
s3_old_shots = s3_data_old[s3_data_old["obj-type"] == "shots"]
s3_new_shots = s3_data_new[s3_data_new["obj-type"] == "shots"]
s3_old_signals = s3_data_old[s3_data_old["obj-type"] == "signals"]
s3_new_signals = s3_data_new[s3_data_new["obj-type"] == "signals"]

In [13]:
s3_new_shots.head()

,worker#,obj-id,open+read-data-time,wrkr-num-objs,mean-obj-time,num-dsets,mean-dset-time,num-workers,obj-type,tot-num-obj,total-runtime
0,1,1160926028,55.792627,1,55.792627,776,0.071898,1,shots,100,55.813563
1,1,1160930031,38.919221,1,38.919221,373,0.104341,2,shots,100,38.939917
2,2,1160930031,22.222220,1,22.222220,432,0.051440,2,shots,100,38.939917
3,1,1160926028,1.163064,1,1.163064,162,0.007179,4,shots,100,1.871618
4,2,1160926028,1.846029,1,1.846029,213,0.008667,4,shots,100,1.871618


In [14]:
s3_new_signals.head()

,worker#,obj-id,open+read-data-time,wrkr-num-objs,mean-obj-time,num-dsets,mean-dset-time,num-workers,obj-type,tot-num-obj,total-runtime
7,1,p_to_v_expr,8.051642,13,0.619357,13,0.619357,8,signals,250,1616.973469
8,2,p_to_v_expr,8.754190,13,0.673399,13,0.673399,8,signals,250,1616.973469
9,3,p_to_v_expr,8.254301,13,0.634946,13,0.634946,8,signals,250,1616.973469
10,4,p_to_v_expr,7.867847,13,0.605219,13,0.605219,8,signals,250,1616.973469
11,5,p_to_v_expr,8.315548,13,0.639658,13,0.639658,8,signals,250,1616.973469


### Total Runtime and Peformance

Total benchmark runtime in the `tot-runtime` column is the elapsed time of the entire benchmark as measured by the main process. The total runtime encompasses:
1. Dividing data access plan across Dask workers and their intialization.
1. All Dask workers completing their jobs.
1. Collecting Dask worker benchmark data.

Below are four DataFrames with total runtimes separated for the signal and shot benchmarks. Their runtimes are so different that there is no point comparing them together. The new DataFrames include several original columns plus a new column `norm-tot-runtime`. This column holds computed performance ratios to the _baseline_ benchmark. The baseline benchmark is one of the available benchmarks selected because it represents the most common set of libhdf5 settings, compute resources, and data access. The baseline benchmarks are:

* S3 files:
    * Reading all signals for a shot: 1 Dask worker
    * Reading all signals for all the shots: 8 Dask worker

In [15]:
old_shots_runtime = s3_old_shots[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
old_shots_runtime["where"] = "Local"
old_shots_runtime["norm-tot-runtime"] = (
    old_shots_runtime.loc[0, "total-runtime"] / old_shots_runtime["total-runtime"]
)

old_signals_runtime = s3_old_signals[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
old_signals_runtime["where"] = "Local"
old_signals_runtime["norm-tot-runtime"] = (
    old_signals_runtime.loc[0, "total-runtime"] / old_signals_runtime["total-runtime"]
)

new_shots_runtime = s3_new_shots[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
new_shots_runtime["where"] = "S3"
new_shots_runtime["norm-tot-runtime"] = (
    new_shots_runtime.loc[0, "total-runtime"] / new_shots_runtime["total-runtime"]
)

new_signals_runtime = s3_new_signals[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
new_signals_runtime["where"] = "S3"
new_signals_runtime["norm-tot-runtime"] = (
    new_signals_runtime.loc[0, "total-runtime"] / new_signals_runtime["total-runtime"]
)

### Reading All Data from a Single Shot File

Plots of performance ratio and runtime when reading all data from a single shot file:

In [7]:
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
(
    old_shots_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * old_shots_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
    * hv.HLine(1).opts(line_width=0.7, color="pink")
    * new_shots_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * new_shots_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
).options(
    legend_position="top_right",
    title="Shot File: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Performance ratio (>1 better)",
    xlim=(0, old_shots_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
) + (
    old_shots_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * old_shots_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
    * new_shots_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * new_shots_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
).options(
    legend_position="bottom_right",
    title="Shot File: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Total runtime / [s]",
    xlim=(0, old_shots_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    show_grid=True,
    height=400,
    width=500,
)

NameError: name 'old_shots_runtime' is not defined

### Reading All Signals from All Shot Files

Plots of performance ratio and runtime when reading all signals from all the shot files in S3:

In [15]:
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
(
    old_signals_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * old_signals_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
    * hv.HLine(1).opts(line_width=0.7, color="pink")
    * new_signals_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * new_signals_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
).options(
    legend_position="top_right",
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Performance ratio (>1 better)",
    xlim=(0, old_signals_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
) + (
    old_signals_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * old_signals_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
    * new_signals_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * new_signals_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
).options(
    legend_position="bottom_right",
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Total runtime / [s]",
    xlim=(0, old_signals_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    show_grid=True,
    height=400,
    width=500,
)

:Layout
   .Overlay.I  :Overlay
      .Curve.I    :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.I  :Scatter   [num-workers]   (norm-tot-runtime)
      .HLine.I    :HLine   [x,y]
      .Curve.II   :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.II :Scatter   [num-workers]   (norm-tot-runtime)
   .Overlay.II :Overlay
      .Curve.I    :Curve   [num-workers]   (total-runtime)
      .Scatter.I  :Scatter   [num-workers]   (total-runtime)
      .Curve.II   :Curve   [num-workers]   (total-runtime)
      .Scatter.II :Scatter   [num-workers]   (total-runtime)

### Display Some Worker Mean Runtimes

The `mean-obj-time` column holds mean read times of _objects_ in a shot file. Which _object_ is read depends on the `obj-type` column, with the values `shots` or `signals`. The `s3_signals` DataFrame 

In [17]:
(
    s3_old_signals.hvplot.box(
        y="mean-obj-time",
        by=["num-workers"],
    )
    * s3_new_signals.hvplot.box(
        y="mean-obj-time",
        by=["num-workers"],
    )
).options(
    # ylim=(8, 11),
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    height=400,
    show_legend=False,
    xlabel="File page buffer size, Number of Dask workers",
    ylabel="Worker mean signal read time / [seconds]",
    show_grid=True,
)

:Overlay
   .BoxWhisker.I  :BoxWhisker   [num-workers]   (mean-obj-time)
   .BoxWhisker.II :BoxWhisker   [num-workers]   (mean-obj-time)

In [18]:
s3_new_signals.groupby(["num-workers"])["mean-obj-time"].describe()

,count,mean,std,min,25%,50%,75%,max
num-workers,,,,,,,,
8,1993.0,0.534262,0.278311,0.220715,0.325284,0.533987,0.645374,3.302288
16,3806.0,0.685725,0.286470,0.231105,0.527410,0.655353,0.801454,4.310197


In [19]:
s3_old_signals.groupby(["num-workers"])["mean-obj-time"].describe()

,count,mean,std,min,25%,50%,75%,max
num-workers,,,,,,,,
8,1993.0,1.644562,0.483214,0.867062,1.364350,1.497909,1.755008,5.432870
16,3806.0,3.984667,2.099014,0.978687,2.769555,3.485612,4.494150,39.513084


In [21]:
s3_old_signals.groupby(["num-workers"]).describe()

worker#                                                 \
              count      mean       std  min  25%  50%   75%   max   
num-workers                                                          
8            1993.0  4.488710  2.287779  1.0  2.0  4.0   6.0   8.0   
16           3806.0  8.130321  4.414745  1.0  4.0  8.0  12.0  16.0   

            open+read-data-time                                             \
                          count       mean        std       min        25%   
num-workers                                                                  
8                        1993.0  20.023365   6.860727  1.734124  16.865377   
16                       3806.0  25.982734  14.966130  1.185076  18.091934   

                                              wrkr-num-objs             \
                   50%        75%         max         count       mean   
num-workers                                                              
8            18.709409  21.995183   69.079080        1993.0  12.149022   
16           23.100851  30.262617  276.591591        3806.0   6.495008   

                                                   mean-obj-time            \
                  std  min   25%   50%   75%   max         count      mean   
num-workers                                                                  
8            1.970253  1.0  12.0  13.0  13.0  14.0        1993.0  1.644562   
16           1.294627  1.0   7.0   7.0   7.0   8.0        3806.0  3.984667   

                                                                          \
                  std       min       25%       50%       75%        max   
num-workers                                                                
8            0.483214  0.867062  1.364350  1.497909  1.755008   5.432870   
16           2.099014  0.978687  2.769555  3.485612  4.494150  39.513084   

            num-dsets                                                    \
                count       mean       std  min   25%   50%   75%   max   
num-workers                                                               
8              1993.0  24.076769  6.338876  2.0  24.0  26.0  26.0  42.0   
16             3806.0  12.869417  3.690671  1.0  12.0  14.0  14.0  21.0   

            mean-dset-time                                                    \
                     count      mean       std       min       25%       50%   
num-workers                                                                    
8                   1993.0  0.870412  0.357284  0.382052  0.679710  0.755181   
16                  3806.0  2.104715  1.239087  0.475741  1.375187  1.777785   

                                 tot-num-obj                                   \
                  75%        max       count   mean  std    min    25%    50%   
num-workers                                                                     
8            0.909712   5.313775      1993.0  250.0  0.0  250.0  250.0  250.0   
16           2.390606  19.756542      3806.0  250.0  0.0  250.0  250.0  250.0   

                          total-runtime                             \
               75%    max         count         mean           std   
num-workers                                                          
8            250.0  250.0        1993.0  4991.579718  0.000000e+00   
16           250.0  250.0        3806.0  6186.241370  9.096142e-13   

                                                                              
                     min          25%          50%          75%          max  
num-workers                                                                   
8            4991.579718  4991.579718  4991.579718  4991.579718  4991.579718  
16           6186.241370  6186.241370  6186.241370  6186.241370  6186.241370

---